In [ ]:
import pandas as pd
import setup_paths
import config
from utils import detect_modalities

### Split file

In [2]:
split_df = pd.read_csv(config.SPLIT_CSV)

# Count subjects per split
split_counts = split_df.count()
print("=== Subjects per split (from CSV) ===")
print(split_counts)

=== Subjects per split (from CSV) ===
train_sessions    664
eval_sessions     141
test_sessions     147
dtype: int64


In [3]:
# Build lookup sets for each split
split_subjects = {}
for col in split_df.columns:
    split_subjects[col] = set(split_df[col].dropna().astype(str).str.strip())

# Scan dataset folders
subject_dirs = [d for d in config.DATA_ROOT.iterdir() if d.is_dir()
                and not d.name.startswith("._")]

print(f"=== Subject folders found in dataset: {len(subject_dirs)} ===")

=== Subject folders found in dataset: 952 ===


In [4]:
# Verify modalities per subject
rows = []

for subject_dir in subject_dirs:
    folder_name = subject_dir.name  # e.g. 002_S_0413_init

    # determine split membership
    split_name = "not_in_split"
    for col, ids in split_subjects.items():
        if folder_name in ids:
            split_name = col
            break

    mods = detect_modalities(subject_dir)

    row = {
        "folder_name": folder_name,
        "split": split_name,
    }

    for mod in config.MODALITIES:
        row[f"{mod}_nii.gz"] = mods[mod]["nii.gz"]
        row[f"{mod}_json"] = mods[mod]["json"]

    rows.append(row)

mods_df = pd.DataFrame(rows)

In [5]:
mods_df

,folder_name,split,T1_nii.gz,T1_json,T2_nii.gz,T2_json,FLAIR_nii.gz,FLAIR_json
0,002_S_0413_init,train_sessions,True,True,True,True,True,True
1,002_S_10814_sc,train_sessions,True,True,True,True,True,True
2,002_S_1155_init,train_sessions,True,True,True,True,True,True
3,002_S_1280_init,test_sessions,True,True,True,True,True,True
4,002_S_4213_init,train_sessions,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...
947,941_S_7046_init,train_sessions,True,True,True,True,True,True
948,941_S_7051_init,train_sessions,True,True,True,True,True,True
949,941_S_7051_m12,train_sessions,True,True,True,True,True,True
950,941_S_7074_init,eval_sessions,True,True,True,True,True,True


In [6]:
# check that all folders contains all modalities
mods_df.iloc[:, 2:].value_counts()

T1_nii.gz  T1_json  T2_nii.gz  T2_json  FLAIR_nii.gz  FLAIR_json
True       True     True       True     True          True          952
Name: count, dtype: int64

In [7]:
# Print split counts found in dataset
print("=== Subject folders matched to splits ===")
print(mods_df["split"].value_counts())

=== Subject folders matched to splits ===
split
train_sessions    664
test_sessions     147
eval_sessions     141
Name: count, dtype: int64


### Meta file

In [8]:
meta_df = pd.read_csv(config.META_CSV)

In [9]:
meta_df

,image_data_id,subject,session_id,group,sex,age,visit,modality,description,type,acq_date,format,mod_family
0,I11128521,002_S_0413,002_S_0413_init,CN,F,95,init,MRI,Sagittal 3D FLAIR (MSV22),Original,2/19/2025,DCM,FLAIR
1,I11128519,002_S_0413,002_S_0413_init,CN,F,95,init,MRI,Accelerated Sagittal MPRAGE (MSV21),Original,2/19/2025,DCM,T1
2,I11128524,002_S_0413,002_S_0413_init,CN,F,95,init,MRI,Sagittal 3D T2 SPACE (MSV21),Original,2/19/2025,DCM,T2_struct
3,I11306628,002_S_10814,002_S_10814_sc,MCI,M,77,sc,MRI,Sagittal 3D FLAIR (MSV22),Original,6/03/2025,DCM,FLAIR
4,I11306627,002_S_10814,002_S_10814_sc,MCI,M,77,sc,MRI,Accelerated Sagittal MPRAGE (MSV21),Original,6/03/2025,DCM,T1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2851,I10283169,941_S_7074,941_S_7074_init,CN,M,72,init,MRI,Accelerated Sagittal MPRAGE (MSV21),Original,9/18/2023,DCM,T1
2852,I10283171,941_S_7074,941_S_7074_init,CN,M,72,init,MRI,Sagittal 3D T2 SPACE (MSV21),Original,9/18/2023,DCM,T2_struct
2853,I11455285,941_S_7074,941_S_7074_m24,CN,M,74,m24,MRI,Sagittal 3D FLAIR (MSV22),Original,10/01/2025,DCM,FLAIR
2854,I11455284,941_S_7074,941_S_7074_m24,CN,M,74,m24,MRI,Accelerated Sagittal MPRAGE (MSV21),Original,10/01/2025,DCM,T1


In [10]:
meta_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2856 entries, 0 to 2855
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   image_data_id  2856 non-null   str  
 1   subject        2856 non-null   str  
 2   session_id     2856 non-null   str  
 3   group          2856 non-null   str  
 4   sex            2856 non-null   str  
 5   age            2856 non-null   int64
 6   visit          2856 non-null   str  
 7   modality       2856 non-null   str  
 8   description    2856 non-null   str  
 9   type           2856 non-null   str  
 10  acq_date       2856 non-null   str  
 11  format         2856 non-null   str  
 12  mod_family     2856 non-null   str  
dtypes: int64(1), str(12)
memory usage: 290.2 KB


In [11]:
rows = []
for split_name in split_df.columns:
    for sid in split_df[split_name].dropna().astype(str):
        rows.append((sid, split_name))

In [12]:
split = pd.DataFrame(rows, columns=["session_id", "split"])
labels = meta_df[["session_id", "group"]].drop_duplicates().rename(columns={"group": "label"})

merged = split.merge(labels, on="session_id", how="left")
label_map = {"CN": 0, "MCI": 1, "AD": 2}
merged["label_num"] = merged["label"].map(label_map)

In [13]:
merged

,session_id,split,label,label_num
0,002_S_0413_init,train_sessions,CN,0
1,002_S_10814_sc,train_sessions,MCI,1
2,002_S_1155_init,train_sessions,MCI,1
3,002_S_4213_init,train_sessions,CN,0
4,002_S_4799_init,train_sessions,MCI,1
...,...,...,...,...
947,941_S_10003_sc,test_sessions,MCI,1
948,941_S_10304_sc,test_sessions,CN,0
949,941_S_10534_sc,test_sessions,CN,0
950,941_S_6384_init,test_sessions,CN,0


#### Check that splits are disjoint

In [14]:
import re

# check all session_ids match the expected pattern
pattern = r'^\d+_S_\d+_+[a-zA-Z0-9]+$'
print(sum(bool(re.match(pattern, subj)) for subj in merged["session_id"]))

952


In [15]:
train_set = set(merged[merged["split"] == 'train_sessions']["session_id"])
eval_set = set(merged[merged["split"] == 'eval_sessions']["session_id"])
test_set = set(merged[merged["split"] == 'test_sessions']["session_id"])

train_set = { "_".join(session_id.split("_")[:3]) for session_id in train_set }
eval_set = { "_".join(session_id.split("_")[:3]) for session_id in eval_set }
test_set = { "_".join(session_id.split("_")[:3]) for session_id in test_set }

# check that there is no overlap between sets
print(train_set & eval_set)
print(train_set & test_set)
print(eval_set & test_set)

set()
set()
set()


#### Count how many subjects per class are in each split

In [16]:
# Training class counts
tot_train = merged[merged["split"] == "train_sessions"].shape[0]
train = pd.DataFrame(merged[merged["split"] == "train_sessions"]["label"].value_counts())
train["%"]=train["count"]/tot_train
train

,count,%
label,,
CN,343,0.516566
MCI,272,0.409639
AD,49,0.073795


In [17]:
# Validation class counts
tot_eval = merged[merged["split"] == "eval_sessions"].shape[0]
eval = pd.DataFrame(merged[merged["split"] == "eval_sessions"]["label"].value_counts())
eval["%"]=eval["count"]/tot_eval
eval

,count,%
label,,
CN,75,0.531915
MCI,54,0.382979
AD,12,0.085106


In [18]:
# Test class counts
tot_test = merged[merged["split"] == "test_sessions"].shape[0]
test = pd.DataFrame(merged[merged["split"] == "test_sessions"]["label"].value_counts())
test["%"]=test["count"]/tot_test
test

,count,%
label,,
CN,72,0.489796
MCI,63,0.428571
AD,12,0.081633


The dataset shows a clear class imbalance, with AD being the smallest class in all splits. This makes balanced evaluation and imbalance-aware training important before building the model.

In [19]:
# Save the split file with labels
# merged[["session_id", "split", "label_num"]].to_csv(config.SPLIT_DIR / "split_labels.csv", index=False)